# Predictive Baseline

This notebook builds a conventional machine-learning model to predict customer renewal.

The model answers:

> Which customers are likely to renew?

It does not directly answer:

> Which customers will renew because they receive the retention offer?

The predictive model will be evaluated using ROC-AUC and Brier score.

In [1]:
from pathlib import Path

import pandas as pd


current_path = Path.cwd().resolve()

if (current_path / "README.md").exists():
    PROJECT_ROOT = current_path
else:
    PROJECT_ROOT = current_path.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_retention_observed.csv"
)

df = pd.read_csv(DATA_PATH)

print("Data path:", DATA_PATH)
print("File exists:", DATA_PATH.exists())
print("Dataset shape:", df.shape)
print("Total missing values:", df.isna().sum().sum())

display(df.head())

Data path: D:\article28-unet-projects\predictive-vs-causal-decision-lab\data\processed\customer_retention_observed.csv
File exists: True
Dataset shape: (6000, 9)
Total missing values: 0


,age,tenure,spend,complaints,satisfaction,engagement,region,treatment,outcome
0,47.656605,39.445856,80.538653,1,0.739655,0.149673,South,0,0
1,31.520191,59.643340,124.772237,0,0.621712,0.503829,West,0,0
2,53.005414,14.470515,55.084554,0,0.748244,0.878509,South,1,0
3,55.286777,29.174800,50.544936,1,0.754126,0.605190,East,1,0
4,20.587578,7.246782,47.698135,2,0.532050,0.356345,North,0,0


## 2. Predictors and Target

The predictive model uses observed customer characteristics and treatment status to predict the observed renewal outcome.

The model does not have access to potential outcomes or true treatment effects.

In [2]:
numeric_features = [
    "age",
    "tenure",
    "spend",
    "complaints",
    "satisfaction",
    "engagement",
    "treatment",
]

categorical_features = [
    "region",
]

predictive_features = (
    numeric_features
    + categorical_features
)

target = "outcome"

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Predictive features:", predictive_features)
print("Target:", target)

print(
    "Number of predictive features:",
    len(predictive_features),
)

Numeric features: ['age', 'tenure', 'spend', 'complaints', 'satisfaction', 'engagement', 'treatment']
Categorical features: ['region']
Predictive features: ['age', 'tenure', 'spend', 'complaints', 'satisfaction', 'engagement', 'treatment', 'region']
Target: outcome
Number of predictive features: 8


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


prepared_transformer = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_features,
        ),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

print(prepared_transformer)

ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 ['age', 'tenure', 'spend', 'complaints',
                                  'satisfaction', 'engagement', 'treatment']),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['region'])],
                  verbose_feature_names_out=False)


## 3. Train-Test Split

The dataset is divided into:

- a training set used to fit the preprocessing steps and predictive model,
- a test set used only for final model evaluation.

The outcome distribution is preserved in both sets using stratified sampling.

In [4]:
from sklearn.model_selection import train_test_split


train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    stratify=df[target],
    random_state=42,
)

print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)

print(
    "Training outcome rate:",
    train_df[target].mean(),
)

print(
    "Test outcome rate:",
    test_df[target].mean(),
)

Training shape: (4500, 9)
Test shape: (1500, 9)
Training outcome rate: 0.45511111111111113
Test outcome rate: 0.4553333333333333


In [5]:
X_train = prepared_transformer.fit_transform(
    train_df[predictive_features]
)

X_test = prepared_transformer.transform(
    test_df[predictive_features]
)

y_train = train_df[target].to_numpy()
y_test = test_df[target].to_numpy()

transformed_feature_names = (
    prepared_transformer.get_feature_names_out()
)

print("Transformed training shape:", X_train.shape)
print("Transformed test shape:", X_test.shape)

print(
    "Transformed feature names:",
    transformed_feature_names.tolist(),
)

print("Training target shape:", y_train.shape)
print("Test target shape:", y_test.shape)

Transformed training shape: (4500, 11)
Transformed test shape: (1500, 11)
Transformed feature names: ['age', 'tenure', 'spend', 'complaints', 'satisfaction', 'engagement', 'treatment', 'region_East', 'region_North', 'region_South', 'region_West']
Training target shape: (4500,)
Test target shape: (1500,)


## 4. Predictive Model and Evaluation

A logistic regression model is trained to predict the observed renewal outcome.

Performance is evaluated using:

- ROC-AUC for ranking customers,
- Brier score for probability accuracy.

These predictive metrics do not measure causal treatment effects.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score


predictive_model = LogisticRegression(
    max_iter=1500,
    random_state=42,
)

predictive_model.fit(
    X_train,
    y_train,
)

renewal_probability = predictive_model.predict_proba(
    X_test
)[:, 1]

roc_auc = roc_auc_score(
    y_test,
    renewal_probability,
)

brier_score = brier_score_loss(
    y_test,
    renewal_probability,
)

print("ROC-AUC:", roc_auc)
print("Brier score:", brier_score)

print(
    "Minimum predicted probability:",
    renewal_probability.min(),
)

print(
    "Maximum predicted probability:",
    renewal_probability.max(),
)

print(
    "Mean predicted probability:",
    renewal_probability.mean(),
)

print(
    "Observed test outcome rate:",
    y_test.mean(),
)

ROC-AUC: 0.6375824132499179
Brier score: 0.23386159336084172
Minimum predicted probability: 0.10145193965217932
Maximum predicted probability: 0.7436670727454459
Mean predicted probability: 0.44828662095313987
Observed test outcome rate: 0.4553333333333333


In [7]:
REPORTS_TABLES_DIR = (
    PROJECT_ROOT
    / "reports"
    / "tables"
)

REPORTS_TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


assert len(renewal_probability) == len(y_test)

assert (
    (renewal_probability >= 0)
    & (renewal_probability <= 1)
).all()

assert 0 <= roc_auc <= 1
assert 0 <= brier_score <= 1


metrics_df = pd.DataFrame(
    {
        "metric": [
            "roc_auc",
            "brier_score",
            "mean_predicted_probability",
            "observed_test_outcome_rate",
        ],
        "value": [
            roc_auc,
            brier_score,
            renewal_probability.mean(),
            y_test.mean(),
        ],
    }
)


metrics_path = (
    REPORTS_TABLES_DIR
    / "predictive_baseline_metrics.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False,
)


display(metrics_df)

print("Metrics saved to:", metrics_path)
print("Metrics file exists:", metrics_path.exists())
print("All predictive validation checks passed.")

,metric,value
0,roc_auc,0.637582
1,brier_score,0.233862
2,mean_predicted_probability,0.448287
3,observed_test_outcome_rate,0.455333


Metrics saved to: D:\article28-unet-projects\predictive-vs-causal-decision-lab\reports\tables\predictive_baseline_metrics.csv
Metrics file exists: True
All predictive validation checks passed.
